# Model Training
## Telco Customer Churn Prediction

Training 6 different models and comparing them to pick the best two for tuning.
No single model is always the best, so it's worth running a few and seeing what the data responds to.

Sections:
1. Load data
2. Train all 6 models
3. Compare results
4. Why did each model perform the way it did?
5. Classification reports for top 3
6. Pick top 2 for hyperparameter tuning

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from src.preprocessing import load_data, clean_data, split_data
from src.features      import engineer_features, create_preprocessor
from src.train         import ModelTrainer

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

## 1. Load and Prepare Data

In [ ]:
RAW_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = engineer_features(clean_data(load_data(RAW_PATH)))
X_train, X_test, y_train, y_test = split_data(df)

# load the already-fitted preprocessor from notebook 02
preprocessor = joblib.load("../models/preprocessor.joblib")
X_train_proc  = preprocessor.transform(X_train)
X_test_proc   = preprocessor.transform(X_test)

print(f"Train: {X_train_proc.shape}  |  churn rate: {y_train.mean():.1%}")
print(f"Test:  {X_test_proc.shape}   |  churn rate: {y_test.mean():.1%}")

## 2. Train All 6 Models

Models: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, SVM.

Each model uses `class_weight='balanced'` (or `scale_pos_weight` for XGBoost) to handle the class imbalance.

In [ ]:
trainer = ModelTrainer(X_train_proc, y_train, X_test_proc, y_test)
comparison_df = trainer.train_all()

In [ ]:
print("\nFinal comparison (sorted by F1):")
display(comparison_df.style
    .background_gradient(subset=["f1", "roc_auc", "recall"], cmap="YlGn")
    .format({c: "{:.4f}" for c in comparison_df.columns if comparison_df[c].dtype == float})
)

## 3. Comparison Charts

In [ ]:
models_order = comparison_df.index.tolist()
metrics_4    = ["accuracy", "precision", "recall", "f1"]
colors_4     = sns.color_palette("Set2", 4)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# grouped bar chart — accuracy, precision, recall, F1 for all models
ax = axes[0, 0]
x  = np.arange(len(models_order))
w  = 0.18
for i, (metric, color) in enumerate(zip(metrics_4, colors_4)):
    vals = [comparison_df.loc[m, metric] for m in models_order]
    ax.bar(x + i*w, vals, width=w, label=metric.title(),
           color=color, edgecolor="white")
ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(models_order, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("All 4 Metrics Across Models", fontweight="bold")
ax.legend(loc="lower right")
ax.axhline(0.73, color="gray", linestyle="--", linewidth=0.8, label="Naive accuracy")

# ROC-AUC
ax = axes[0, 1]
roc_vals   = [comparison_df.loc[m, "roc_auc"] for m in models_order]
colors_roc = sns.color_palette("viridis", len(models_order))
bars = ax.bar(models_order, roc_vals, color=colors_roc, edgecolor="white")
for bar, val in zip(bars, roc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{val:.3f}", ha="center", fontweight="bold", fontsize=9)
ax.set_xticklabels(models_order, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.7, 1.0)
ax.set_title("ROC-AUC Scores", fontweight="bold")

# training time
ax = axes[1, 0]
times = [comparison_df.loc[m, "train_time_seconds"] for m in models_order]
bars  = ax.barh(models_order[::-1], times[::-1],
                color=sns.color_palette("Set2", len(models_order)), edgecolor="white")
for bar, val in zip(bars, times[::-1]):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f"{val:.1f}s", va="center", fontsize=9)
ax.set_xlabel("Training Time (seconds)")
ax.set_title("Training Time", fontweight="bold")

# F1 vs training time (efficiency plot)
ax = axes[1, 1]
scatter_colors = sns.color_palette("tab10", len(models_order))
for i, m in enumerate(models_order):
    ax.scatter(comparison_df.loc[m, "train_time_seconds"],
               comparison_df.loc[m, "f1"],
               s=150, color=scatter_colors[i], zorder=3, label=m)
    ax.annotate(m, xy=(comparison_df.loc[m, "train_time_seconds"],
                       comparison_df.loc[m, "f1"]),
                xytext=(5, 3), textcoords="offset points", fontsize=8)
ax.set_xlabel("Training Time (seconds)")
ax.set_ylabel("F1 Score")
ax.set_title("F1 vs Training Time (top-right = most efficient)", fontweight="bold")
ax.legend(fontsize=7, loc="lower right")

plt.suptitle("Model Comparison", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_11_model_comparison.png", bbox_inches="tight", dpi=150)
plt.show()

## 4. Why Did Each Model Perform This Way?

| Model | What to expect | Why |
|-------|---------------|-----|
| Logistic Regression | Strong baseline | If LR does well it means the features have linear separability |
| Decision Tree | Weaker than ensembles | A single tree overfits easily — high variance |
| Random Forest | Better than DT | Averaging 200 trees reduces variance |
| Gradient Boosting | Very strong | Iteratively corrects the errors of previous trees |
| XGBoost | Best or near-best | Built-in L1/L2 regularisation, handles tabular data well |
| SVM | Mid-range | Powerful with small feature sets but slower after OHE expansion |

The most important metric for this problem is **recall on the Churn class** — we want to catch as many churners as possible. Missing a churner (false negative) is more costly than a false alarm (false positive) because a missed churner means lost revenue with no chance to intervene.

## 5. Classification Reports — Top 3 Models

In [ ]:
trainer.print_classification_reports(top_n=3)

## 6. Pick Top 2 and Save All Models

In [ ]:
top2 = trainer.get_top_n_models(n=2, metric="f1")

print("Top 2 models for hyperparameter tuning:")
for i, name in enumerate(top2, 1):
    row = comparison_df.loc[name]
    print(f"  {i}. {name:25s}  F1={row['f1']:.4f}  ROC-AUC={row['roc_auc']:.4f}")

print(f"\nTaking {top2[0]} and {top2[1]} into notebook 04 for tuning.")

print("\nSaving baseline models...")
trainer.save_models(directory="../models/baseline_models")
print("Done.")